In [73]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    noise: float = 0.0
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        result = []
        action_keys = list(self.actions.keys())
        aind = list(self.actions.keys()).index(action)
        n_actions = len(action_keys)
        for a, prob in zip(
            [aind, (aind + 1) % n_actions, (aind - 1) % n_actions],
            [1 - self.noise, self.noise / 2, self.noise / 2],
        ):
            movement = self.actions[action_keys[a]]
            next_cell = (cell[0] + movement[0], cell[1] + movement[1])
            outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
            on_wall = next_cell in self.walls
            if outside or on_wall:
                next_cell = cell

            if next_cell in self.terminals:
                reward = self.terminals[next_cell]
            else:
                reward = self.step_reward
            result.append((prob, next_cell, reward))
        return result
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
    noise={self.noise},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=-0.04,
    terminals={(0, 3): 1},
    walls={(1, 1)},
    noise=0.2,
)
default_grid

Grid(rows=3, cols=4, step_reward=-0.04, terminals={(0, 3): 1}, walls={(1, 1)}, noise=0.2)

In [74]:
default_grid.render()

 r/c  0   1   2   3 
  0   ·   ·   ·   1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [75]:
class Sampler:
    def __init__(self, env: Grid, rng: np.random.Generator):
        self.env = env
        self.rng = rng
        self.empty_cells = [
            (r, c)
            for r in range(env.rows)
            for c in range(env.cols)
            if (r, c) not in env.terminals and (r, c) not in env.walls
        ]
        self.nS = len(self.empty_cells)

    def reset(self):
        i = self.rng.choice(len(self.empty_cells))
        return self.empty_cells[i]
    def step(self, cell, action):
        result = self.env.step(cell, action)
        i = self.rng.choice(len(result), p=[p for p, _, _ in result])
        _, next_cell, reward = result[i]
        done = True if next_cell in self.env.terminals else False
        return next_cell, reward, done

In [76]:
sampler = Sampler(default_grid, np.random.default_rng(0))
print(sampler.reset())

(2, 2)


In [77]:
print(sampler.step((0, 0), 'right'))
print(sampler.step((0, 2), 'right'))

((0, 1), -0.04, False)
((0, 3), 1, True)


In [78]:
cell = (1, 2)
action = 'up'
iters = 1000
counts = {}
for _ in range(iters):
    next_cell, _, _ = sampler.step(cell, action)
    counts[next_cell] = counts.get(next_cell, 0) + 1
print(counts)
print(
    [(next_cell, f"{(count / iters * 100):.2f}") 
     for next_cell, count in counts.items()]
)

{(0, 2): 787, (1, 3): 117, (1, 2): 96}
[((0, 2), '78.70'), ((1, 3), '11.70'), ((1, 2), '9.60')]


In [79]:
counts = {}
for _ in range(iters):
    start_cell = sampler.reset()
    counts[start_cell] = counts.get(start_cell, 0) + 1
print(len(counts))
print(counts)
print(
    [
        (start_cell, f"{(count / iters * 100):.2f}")
        for start_cell, count in counts.items()
    ]
)

10
{(2, 0): 101, (1, 0): 101, (1, 2): 78, (1, 3): 102, (2, 3): 120, (2, 2): 109, (2, 1): 89, (0, 2): 89, (0, 0): 112, (0, 1): 99}
[((2, 0), '10.10'), ((1, 0), '10.10'), ((1, 2), '7.80'), ((1, 3), '10.20'), ((2, 3), '12.00'), ((2, 2), '10.90'), ((2, 1), '8.90'), ((0, 2), '8.90'), ((0, 0), '11.20'), ((0, 1), '9.90')]


In [80]:
import io, contextlib, functools

def silent(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return fn(*args, **kwargs)
    return wrapper

In [81]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if r == 0 and c == 0:
                print("r/c", end="")
                print(''.join([f'{v:>2} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>2} ', end="")

            if policy[r][c] != None:
                value = grid.arrows[max(policy[r][c], key=policy[r][c].get)]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')

def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma=0.9, theta=1e-6, max_iters=1000):
    print('---------- value_iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                def q(action):
                    result = grid.step(cell, action)
                    q_value = 0
                    for prob, next_cell, reward in result:
                        q_value += prob * (
                            reward + gamma * V_old[next_cell[0]][next_cell[1]]
                        )
                    return q_value
                new_value = float('-inf')
                for action in grid.actions:
                    value = q(action)
                    if value > new_value:
                        new_value = value
                        policy[r][c] = {action: 1.0}
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    render_policy(grid, policy)
    converged = delta <= theta
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [82]:
policy, V, converged = value_iteration(default_grid);

---------- value_iteration ------------
  r/c   0    1    2    3 
   0 -0.04 -0.04 0.79    0 
   1 -0.04    0 -0.04 0.79 
   2 -0.04 -0.04 -0.04 -0.04 
------------------
  r/c   0    1    2    3 
   0 -0.08 0.52 0.86    0 
   1 -0.08    0  0.6 0.86 
   2 -0.08 -0.08 -0.08 0.52 
------------------
  r/c   0    1    2    3 
   0 0.32 0.67 0.92    0 
   1 -0.11    0 0.71 0.92 
   2 -0.11 -0.11 0.43 0.62 
------------------
  r/c   0    1    2    3 
   0 0.46 0.75 0.94    0 
   1 0.17    0 0.77 0.94 
   2 -0.14 0.25 0.52 0.72 
------------------
  r/c   0    1    2    3 
   0 0.55 0.77 0.95    0 
   1 0.33    0 0.79 0.95 
   2 0.14 0.38  0.6 0.75 
------------------
  r/c   0    1    2    3 
   0 0.59 0.78 0.95    0 
   1 0.42    0  0.8 0.95 
   2 0.27 0.46 0.63 0.76 
------------------
  r/c   0    1    2    3 
   0 0.61 0.78 0.95    0 
   1 0.46    0  0.8 0.95 
   2 0.35  0.5 0.64 0.77 
------------------
  r/c   0    1    2    3 
   0 0.62 0.78 0.95    0 
   1 0.48    0  0.8 0.95 
   2

In [83]:
def read_policy(grid: Grid, V, gamma=0.9, incumbent_policy=None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            def q(action):
                result = grid.step(cell, action)
                q_value = 0
                for prob, next_cell, reward in result:
                    q_value += prob * (
                        reward + gamma * V[next_cell[0]][next_cell[1]]
                    )
                return q_value
            new_action = max(grid.actions, key=q)
            if incumbent_policy:
                incumbent_action = max(incumbent_policy[r][c], key=incumbent_policy[r][c].get)
                if q(new_action) - q(incumbent_action) < 1e-9:
                    new_action = incumbent_action
            policy[r][c] = {new_action: 1.0}
    return policy


policy = read_policy(default_grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, {'up': 1.0}],
 [{'right': 1.0}, {'right': 1.0}, {'up': 1.0}, {'up': 1.0}]]

In [84]:
render_policy(default_grid, policy)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------


In [85]:
def policy_evaluation(
    grid: Grid, policy, gamma=0.9, theta=1e-6, verbose=False, max_iters=1000
):
    if verbose:
        print('---------- policy_evaluation ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                value = 0
                def q(action):
                    result = grid.step(cell, action)
                    q_value = 0
                    for prob, next_cell, reward in result:
                        q_value += prob * (
                            reward + gamma * V_old[next_cell[0]][next_cell[1]]
                        )
                    return q_value
                for action, prob in policy[r][c].items():
                    value += prob * q(action)
                V[r][c] = value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
        i += 1
    converged = delta <= theta
    if converged:
        print(f'policy_evaluation converged in {i} iterations')
    else:
        print(f'policy_evaluation did not converge in {max_iters} iterations')
    return V, converged


policy_evaluation(default_grid, policy, verbose=True);

---------- policy_evaluation ------------
  r/c   0    1    2    3 
   0 -0.04 -0.04 0.79    0 
   1 -0.04    0 -0.04 0.79 
   2 -0.04 -0.04 -0.04 -0.04 
------------------
  r/c   0    1    2    3 
   0 -0.08 0.52 0.86    0 
   1 -0.08    0  0.6 0.86 
   2 -0.08 -0.08 -0.08 0.52 
------------------
  r/c   0    1    2    3 
   0 0.32 0.67 0.92    0 
   1 -0.11    0 0.71 0.92 
   2 -0.11 -0.11 0.43 0.62 
------------------
  r/c   0    1    2    3 
   0 0.46 0.75 0.94    0 
   1 0.17    0 0.77 0.94 
   2 -0.14 0.25 0.52 0.72 
------------------
  r/c   0    1    2    3 
   0 0.55 0.77 0.95    0 
   1 0.33    0 0.79 0.95 
   2 0.14 0.38  0.6 0.75 
------------------
  r/c   0    1    2    3 
   0 0.59 0.78 0.95    0 
   1 0.42    0  0.8 0.95 
   2 0.27 0.46 0.63 0.76 
------------------
  r/c   0    1    2    3 
   0 0.61 0.78 0.95    0 
   1 0.46    0  0.8 0.95 
   2 0.35  0.5 0.64 0.77 
------------------
  r/c   0    1    2    3 
   0 0.62 0.78 0.95    0 
   1 0.48    0  0.8 0.95 
  

In [86]:
def eps_soft(grid: Grid, policy, eps=0.1):
    policy = [row[:] for row in policy]
    action_keys = list(grid.actions.keys())
    n_actions = len(action_keys)
    for r in range(grid.rows):
        for c in range(grid.cols):
            if policy[r][c] is None:
                continue
            ga = max(policy[r][c], key=policy[r][c].get)
            other_actions = {
                action: eps / n_actions
                for action in action_keys
                if action != ga
            }
            greedy_action = {
                ga: 1.0 - eps + eps / n_actions,
            }
            policy[r][c] = {**greedy_action, **other_actions}

    return policy

In [87]:
def policy_iteration(
    grid: Grid,
    policy=None,
    max_iters=1000,
    pass_incumbent_policy=False,
    gamma=0.9,
    theta=1e-6,
    policy_evaluation_verbose=False,
    policy_evaluation_max_iters=1000,
    eps = None,
):
    print('---------- policy_iteration ------------')
    if policy is None:
        policy = [[{'up': 1.0} for _ in range(grid.cols)] for _ in range(grid.rows)]
    i = 0
    policy_evaluation_converged = True
    changed = None
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    while changed != 0 and i < max_iters:
        V, policy_evaluation_converged = policy_evaluation(
            grid,
            policy if eps is None else eps_soft(grid, policy, eps),
            gamma,
            theta,
            verbose=policy_evaluation_verbose,
            max_iters=policy_evaluation_max_iters,
        )
        if not policy_evaluation_converged:
            break
        show_V(grid, V)
        new_policy = read_policy(
            grid, V, gamma, incumbent_policy=policy if pass_incumbent_policy else None
        )
        changed = sum(
            new_policy[r][c] != None
            and (
                max(new_policy[r][c], key=new_policy[r][c].get)
                != max(policy[r][c], key=policy[r][c].get)
            )
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if not policy_evaluation_converged:
        print(
            f"policy_iteration did not converge because policy_evaluation did not converge"
        )
    elif converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [88]:
policy_iteration(default_grid);

---------- policy_iteration ------------
policy_evaluation converged in 82 iterations
  r/c   0    1    2    3 
   0 -0.3 -0.18 0.17    0 
   1 -0.31    0 0.18 0.89 
   2 -0.31 -0.2 0.13 0.67 
------------------
actions changed = 7
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  →  ↑ 
 2  →  →  →  ↑ 
------------------
policy_evaluation converged in 22 iterations
  r/c   0    1    2    3 
   0 0.63 0.78 0.95    0 
   1  0.5    0 0.79 0.95 
   2 0.41 0.52 0.64 0.77 
------------------
actions changed = 1
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  →  ↑ 
------------------
policy_evaluation converged in 20 iterations
  r/c   0    1    2    3 
   0 0.63 0.78 0.95    0 
   1  0.5    0  0.8 0.95 
   2 0.42 0.52 0.65 0.77 
------------------
actions changed = 1
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
policy_evaluation converged in 20 iterations
  r/c   0    1    2    3 
   0 0.63 0.78 0.95    0 
   1  0.5    0  0.8 0.95 
   2 0.42 0.53 0.65 

In [89]:
def generate_episode(
    sampler: Sampler, policy, rng: np.random.Generator, max_steps=1000
):
    cell = sampler.reset()
    done = False
    episode = []
    i = 0
    while not done and i < max_steps:
        (r, c) = cell
        actions = list(policy[r][c])
        a_idx = rng.choice(len(actions), p=[policy[r][c][a] for a in actions])
        action = actions[a_idx]
        next_cell, reward, done = sampler.step(cell, action)
        episode.append((cell, reward, next_cell, done))
        cell = next_cell
        i += 1
    return episode

In [90]:
generate_episode(sampler, policy, np.random.default_rng(1))

[((0, 0), -0.04, (0, 1), False),
 ((0, 1), -0.04, (0, 2), False),
 ((0, 2), 1, (0, 3), True)]

In [91]:
generate_episode(sampler, policy, np.random.default_rng(3))

[((0, 2), 1, (0, 3), True)]

In [92]:
episode = generate_episode(sampler, policy, np.random.default_rng(3))
episode

[((2, 1), -0.04, (2, 2), False),
 ((2, 2), -0.04, (1, 2), False),
 ((1, 2), -0.04, (1, 3), False),
 ((1, 3), 1, (0, 3), True)]

In [93]:
def returns_from_episode(episode, gamma = 0.9):
    G = 0.0
    out = [0.0] * len(episode)
    for t in range(len(episode) - 1, -1, -1):
        _, reward, *_ = episode[t]
        G = reward + gamma * G
        out[t] = G
    return out

In [94]:
gamma = 0.9
returns = returns_from_episode(episode, gamma=gamma)
returns

[0.6205999999999999, 0.734, 0.86, 1.0]

In [95]:
rewards = [r for (_s, r, _ns, _d) in episode]
G0_brute = sum(gamma ** k * rewards[k] for k in range(len(rewards)))
print(G0_brute)

for t in range(len(episode)):
    brute = sum(gamma**(k - t) * rewards[k] for k in range(t, len(rewards)))
    assert np.isclose(returns[t], brute), t


0.6206


In [96]:
def mc_prediction(
    sampler: Sampler,
    policy,
    gamma,
    num_episodes,
    rng: np.random.Generator,
):
    returns_sum = [
        [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
    ]
    returns_cnt = [
        [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
    ]
    for _ in range(num_episodes):
        ep = generate_episode(sampler, policy, rng)
        G = returns_from_episode(ep, gamma)
        seen = set()
        for t, (cell, _r, _ns, _d) in enumerate(ep):
            if cell in seen:
                continue
            seen.add(cell)
            r, c = cell
            returns_sum[r][c] += G[t]
            returns_cnt[r][c] += 1
    V = [
        [
            returns_sum[r][c] / (returns_cnt[r][c] + 1e-9)
            for c in range(sampler.env.cols)
        ]
        for r in range(sampler.env.rows)
    ]
    return V

In [97]:
mc_prediction(sampler, policy, gamma, 1000, np.random.default_rng(3))

[[0.6369131143909661, 0.7930005859426136, 0.9452263466839367, 0.0],
 [0.5163211995234939, 0.0, 0.7991062397291564, 0.948105726615295],
 [0.4036042582306148,
  0.5142136323383011,
  0.6454795865985357,
  0.7622131882454638]]

In [98]:
def rms(sampler: Sampler, V_hat, V_true):
    d = np.array(
        [
            V_hat[r][c] - V_true[r][c]
            for r in range(sampler.env.rows)
            for c in range(sampler.env.cols)
            if (r, c) in sampler.empty_cells
        ]
    )
    return float(np.sqrt(np.mean(d**2)))

In [99]:
V_true, _ = policy_evaluation(default_grid, policy)
rng = np.random.default_rng(3)
for num_episodes in [100, 500, 2000]:
    V = mc_prediction(sampler, policy, gamma, num_episodes, rng)
    error = rms(sampler, V, V_true)
    print(f'{num_episodes=}, {error=}')

policy_evaluation converged in 20 iterations
num_episodes=100, error=0.04070331171902858
num_episodes=500, error=0.0074853301856398205
num_episodes=2000, error=0.0077534048013319055


In [100]:
V_true, _ = policy_evaluation(default_grid, policy)
rng = np.random.default_rng(3)
returns_sum = [[0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)]
returns_cnt = [
    [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
]
checkpoints = [100, 500, 2000, 10000, 50000]
for i in range(1, 50000 + 1):
    ep = generate_episode(sampler, policy, rng)
    G = returns_from_episode(ep, gamma)
    seen = set()
    for t, (cell, _r, _ns, _d) in enumerate(ep):
        if cell in seen:
            continue
        seen.add(cell)
        r, c = cell
        returns_sum[r][c] += G[t]
        returns_cnt[r][c] += 1
    if i in checkpoints:
        V = [
            [returns_sum[r][c] / (returns_cnt[r][c] + 1e-9) for c in range(sampler.env.cols)]
            for r in range(sampler.env.rows)
        ]
        error = rms(sampler, V, V_true)
        print(f'num_episodes={i}, {error=}')

policy_evaluation converged in 20 iterations
num_episodes=100, error=0.03582530043685871
num_episodes=500, error=0.01750448763382791
num_episodes=2000, error=0.0053681479185050335
num_episodes=10000, error=0.0018542678953346822
num_episodes=50000, error=0.001126818599017506


In [101]:
def td0_prediction(
    sampler: Sampler,
    policy,
    gamma,
    num_episodes,
    alpha,
    rng: np.random.Generator,
):
    V = [
        [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
    ]
    cnt = [
        [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
    ]
    for _ in range(num_episodes):
        for cell, reward, next_cell, done in generate_episode(sampler, policy, rng):
            r, c = cell
            cnt[r][c] += 1
            step = alpha if alpha is not None else 1.0 / cnt[r][c]
            target = reward + gamma * V[next_cell[0]][next_cell[1]] * (1.0 - done)
            V[r][c] += step * (target - V[r][c])
    return V

In [102]:
td0_prediction(sampler, policy, gamma, 1000, 0.05, np.random.default_rng(3))

[[0.6134478053095368, 0.782312142835057, 0.9345964755831776, 0],
 [0.47257740939385107, 0, 0.7662628282122215, 0.9551481338236408],
 [0.377644856198654,
  0.5254290754561718,
  0.662586130751186,
  0.7852402460262363]]

In [103]:
V_true, _ = policy_evaluation(default_grid, policy)
# alpha = 0.05
alpha = None
rng = np.random.default_rng(3)
for num_episodes in [100, 500, 2000, 10000, 50000]:
    V = td0_prediction(sampler, policy, gamma, num_episodes, alpha, rng)
    error = rms(sampler, V, V_true)
    print(f'{num_episodes=}, {error=}')

policy_evaluation converged in 20 iterations
num_episodes=100, error=0.1955234513389847
num_episodes=500, error=0.06166731914806009
num_episodes=2000, error=0.06814266752705155
num_episodes=10000, error=0.01538600940667658
num_episodes=50000, error=0.005774348166837911


In [104]:
V_true, _ = policy_evaluation(default_grid, policy)
# alpha = 0.05
alpha = None
rng = np.random.default_rng(3)
V = [
    [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
]
cnt = [
    [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
]
checkpoints = [100, 500, 2000, 10000, 50000]
for i in range(1, 50000 + 1):
    for cell, reward, next_cell, done in generate_episode(sampler, policy, rng):
        r, c = cell
        cnt[r][c] += 1
        step = alpha if alpha is not None else 1.0 / cnt[r][c]
        target = reward + gamma * V[next_cell[0]][next_cell[1]] * (1.0 - done)
        V[r][c] += step * (target - V[r][c])
    if i in checkpoints:
        error = rms(sampler, V, V_true)
        print(f'num_episodes={i}, {error=}')

policy_evaluation converged in 20 iterations
num_episodes=100, error=0.2488266051834627
num_episodes=500, error=0.14003080935222004
num_episodes=2000, error=0.07714139040094603
num_episodes=10000, error=0.03777869757013441
num_episodes=50000, error=0.017454485617272215


In [110]:
def get_checkpoints_track(td_alphas, seed=0):

    V_true, _ = policy_evaluation(default_grid, policy)

    checkpoints = [100, 500, 2000, 10000, 50000]
    track = {}
    # Monte-Carlo
    sampler = Sampler(default_grid, np.random.default_rng(seed))
    rng = np.random.default_rng(seed + 1)
    returns_sum = [
        [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
    ]
    returns_cnt = [
        [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
    ]
    
    for i in range(1, 50000 + 1):
        ep = generate_episode(sampler, policy, rng)
        G = returns_from_episode(ep, gamma)
        seen = set()
        for t, (cell, _r, _ns, _d) in enumerate(ep):
            if cell in seen:
                continue
            seen.add(cell)
            r, c = cell
            returns_sum[r][c] += G[t]
            returns_cnt[r][c] += 1
        if i in checkpoints:
            V = [
                [returns_sum[r][c] / (returns_cnt[r][c] + 1e-9) for c in range(sampler.env.cols)]
                for r in range(sampler.env.rows)
            ]
            error = rms(sampler, V, V_true)
            key = 'MC (1/n)'
            if key not in track:
                track[key] = []
            track[key].append(error)
    # Td0
    for alpha in td_alphas:
        sampler = Sampler(default_grid, np.random.default_rng(seed))
        rng = np.random.default_rng(seed + 1)
        V = [
            [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
        ]
        cnt = [
            [0 for _ in range(sampler.env.cols)] for _ in range(sampler.env.rows)
        ]
        for i in range(1, 50000 + 1):
            for cell, reward, next_cell, done in generate_episode(sampler, policy, rng):
                r, c = cell
                cnt[r][c] += 1
                step = alpha if alpha is not None else 1.0 / cnt[r][c]
                target = reward + gamma * V[next_cell[0]][next_cell[1]] * (1.0 - done)
                V[r][c] += step * (target - V[r][c])
            if i in checkpoints:
                error = rms(sampler, V, V_true)
                key = f'TD a={alpha if alpha is not None else '1/n'}'
                if key not in track:
                    track[key] = []
                track[key].append(error)     
    return checkpoints, track

In [115]:
checkpoints, track = get_checkpoints_track(td_alphas=[None, 0.05], seed=1)

policy_evaluation converged in 20 iterations


In [117]:
def display_track(checkpoints, track):
    print('episodes ' + ' '.join(track.keys()))
    for i, num_episodes in enumerate(checkpoints):
        print(
            f"{num_episodes:>6}{''.join(f'{errors[i]:>10.4f}' for errors in track.values())}"
        )
display_track(checkpoints, track)

episodes MC (1/n) TD a=1/n TD a=0.05
   100    0.0364    0.1630    0.4155
   500    0.0172    0.0951    0.0730
  2000    0.0037    0.0505    0.0118
 10000    0.0035    0.0235    0.0130
 50000    0.0009    0.0103    0.0095


In [114]:
checkpoints, track = get_checkpoints_track(td_alphas=[0.2, 0.05, 0.01], seed=1)
display_track(checkpoints, track)

policy_evaluation converged in 20 iterations
episodes MC (1/n) TD a=0.2 TD a=0.05 TD a=0.01
   100    0.0364    0.1199    0.4155    0.6255
   500    0.0172    0.0248    0.0730    0.4179
  2000    0.0037    0.0282    0.0118    0.1074
 10000    0.0035    0.0232    0.0130    0.0051
 50000    0.0009    0.0163    0.0095    0.0057


**Read your table along a row, not just down a column** — that's where the dial is:

    episodes   a=0.2    a=0.05   a=0.01
         100  0.1114   0.4213   0.6298    <- biggest alpha wins by 6x
       50000  0.0288   0.0159   0.0082    <- smallest alpha wins by 3.5x
    

At 100 episodes α=0.2 has actually moved `V` off zero while α=0.01 has barely started. At 50000 that reverses completely. Fast start versus low floor, one parameter, and no setting is good at both ends.

That's also the argument for a decaying step: the `TD a=1/n` column from the previous table keeps falling with no floor at all.

### batch A/B example

In [118]:
A, B = 0, 1
_AB_EPISODES = (
    [[(A, 0.0, B, False), (B, 0.0, -1, True)]]
    + [[(B, 1.0, -1, True)]] * 6
    + [[(B, 0.0, -1, True)]]
)
def batch_mc(episodes, gamma=1.0):
    """Batch (every-visit) MC fixed point: V(s) = mean observed return from s."""
    rsum, rcnt = {}, {}
    for ep in episodes:
        G = 0.0
        for t in range(len(ep) - 1, -1, -1):
            s, r = ep[t][0], ep[t][1]
            G = r + gamma * G
            rsum[s] = rsum.get(s, 0.0) + G
            rcnt[s] = rcnt.get(s, 0) + 1
    return {s: rsum[s] / rcnt[s] for s in rsum}


def batch_td(episodes, gamma=1.0, alpha=0.01, sweeps=20000):
    """Batch TD(0) fixed point: present all transitions repeatedly until V stops
    moving. Converges to the value of the MAXIMUM-LIKELIHOOD MDP built from the
    data (certainty equivalence)."""
    V = {A: 0.0, B: 0.0}
    for _ in range(sweeps):
        for ep in episodes:
            for (s, r, ns, done) in ep:
                target = r + gamma * (0.0 if done else V[ns])
                V[s] += alpha * (target - V[s])
    return V

In [119]:
vmc = batch_mc(_AB_EPISODES)
vtd = batch_td(_AB_EPISODES)
print("       V(A)     V(B)")
print(f"MC :  {vmc[A]:+.3f}   {vmc[B]:+.3f}     <- mean return seen from each state")
print(f"TD :  {vtd[A]:+.3f}   {vtd[B]:+.3f}     <- value of the implied (ML) MDP")

       V(A)     V(B)
MC :  +0.000   +0.750     <- mean return seen from each state
TD :  +0.750   +0.750     <- value of the implied (ML) MDP


```-
from A:  1 transition,  always to B,        reward 0      -> P(B|A) = 1,   R = 0
from B:  8 transitions, always to terminal, rewards
         {0, 1,1,1,1,1,1, 0}                              -> mean R = 6/8 = 0.75

solve that MDP:   V(B) = R(B) = 0.75
                  V(A) = R(A) + P(B|A) * V(B) = 0 + 1 * 0.75
```

In [120]:
for alpha in [0.001, 0.01, 0.2, 0.5]:
    vtd = batch_td(_AB_EPISODES, alpha=alpha)
    print(f'alpha - {alpha}: {vtd[A]:+.3f}   {vtd[B]:+.3f}')

alpha - 0.001: +0.750   +0.750
alpha - 0.01: +0.750   +0.750
alpha - 0.2: +0.709   +0.709
alpha - 0.5: +0.494   +0.494


In [124]:
_AB_EPISODES = (
    [[(B, 0.0, -1, True)]]
    + [[(B, 1.0, -1, True)]] * 6
    + [[(A, 0.0, B, False), (B, 0.0, -1, True)]]
)
for alpha in [0.001, 0.01, 0.2, 0.5]:
    vtd = batch_td(_AB_EPISODES, alpha=alpha)
    print(f'alpha - {alpha}: {vtd[A]:+.3f}   {vtd[B]:+.3f}')

alpha - 0.001: +0.751   +0.750
alpha - 0.01: +0.757   +0.750
alpha - 0.2: +0.887   +0.709
alpha - 0.5: +0.988   +0.494


In [125]:
_AB_EPISODES = (
    [[(B, 0.0, -1, True)]]
    + [[(A, 0.0, B, False), (B, 0.0, -1, True)]]
    + [[(B, 1.0, -1, True)]] * 6
)
for alpha in [0.001, 0.01, 0.2, 0.5]:
    vtd = batch_td(_AB_EPISODES, alpha=alpha)
    print(f'alpha - {alpha}: {vtd[A]:+.3f}   {vtd[B]:+.3f}')

alpha - 0.001: +0.750   +0.751
alpha - 0.01: +0.750   +0.757
alpha - 0.2: +0.709   +0.887
alpha - 0.5: +0.494   +0.988


In [128]:
def batch_td_sync(episodes, gamma=1.0, alpha=0.01, sweeps=20000):
    """Batch TD(0) fixed point: present all transitions repeatedly until V stops
    moving. Converges to the value of the MAXIMUM-LIKELIHOOD MDP built from the
    data (certainty equivalence)."""
    V = {A: 0.0, B: 0.0}
    for _ in range(sweeps):
        V_old = {**V}
        for ep in episodes:
            for (s, r, ns, done) in ep:
                target = r + gamma * (0.0 if done else V_old[ns])
                V[s] += alpha * (target - V_old[s])
    return V

In [129]:
_AB_EPISODES = (
    [[(A, 0.0, B, False), (B, 0.0, -1, True)]]
    + [[(B, 1.0, -1, True)]] * 6
    + [[(B, 0.0, -1, True)]]
)
for alpha in [0.001, 0.01, 0.2, 0.5]:
    vtd = batch_td_sync(_AB_EPISODES, alpha=alpha)
    print(f'alpha - {alpha}: {vtd[A]:+.3f}   {vtd[B]:+.3f}')

alpha - 0.001: +0.750   +0.750
alpha - 0.01: +0.750   +0.750
alpha - 0.2: +0.750   +0.750
alpha - 0.5: +nan   +nan


In [130]:
_AB_EPISODES = (
    [[(B, 0.0, -1, True)]]
    + [[(A, 0.0, B, False), (B, 0.0, -1, True)]]
    + [[(B, 1.0, -1, True)]] * 6
)
for alpha in [0.001, 0.01, 0.2, 0.5]:
    vtd = batch_td_sync(_AB_EPISODES, alpha=alpha)
    print(f'alpha - {alpha}: {vtd[A]:+.3f}   {vtd[B]:+.3f}')

alpha - 0.001: +0.750   +0.750
alpha - 0.01: +0.750   +0.750
alpha - 0.2: +0.750   +0.750
alpha - 0.5: +nan   +nan


###  what is - certainty-equivalence

It's a principle from stochastic control: **replace the unknown model with your best point estimate, then solve as if that estimate were certain.** You throw away the uncertainty in the estimate and treat it as truth — hence "equivalent to certainty."

In RL it's a two-step recipe:

1.  **Build the maximum-likelihood MDP from the data.** Transition probabilities by counting, rewards by averaging:
    
        P_hat(s'|s,a) = count(s,a,s') / count(s,a)
        R_hat(s,a)    = mean reward observed on (s,a)
        
    
2.  **Solve that MDP exactly** — the same DP you wrote in the other notebook, just pointed at the estimated model instead of the true one.

The result is the certainty-equivalent value estimate. On your A/B data: `P_hat(B|A) = 1`, `R_hat(B) = 6/8`, solve → `V(B)=0.75`, `V(A)=0.75`.

**The theorem is that batch TD(0) lands exactly there without ever building the model.** It never stores counts or probabilities, but its fixed point is identical to solving the counted model. That's what makes TD implicitly model-based. Batch MC does something different — it minimizes squared error against the _observed returns_, which is a fit to the training set, so it says `V(A)=0`.

**The catch, and it's the interesting part.** Certainty equivalence ignores how confident you should be. `P_hat(B|A) = 1.0` is stated with total confidence off a **single** observation. If A actually leads somewhere terrible 30% of the time, the ML model has no idea and your value estimate is confidently wrong. That's the failure mode this principle has, and it's precisely why the exploration literature exists — R-max, UCB, optimistic initialization, posterior sampling are all deliberately _not_ certainty-equivalent. They inflate the value of under-sampled transitions instead of trusting the point estimate.

So: certainty equivalence is the right thing to do with enough data, and a systematic overconfidence bug without it. The gap between those two regimes is the exploration problem, which is what `q_learning_sarsa` runs into next.